In [1]:
from __future__ import annotations
import operator
from pathlib import Path
from typing import List, Literal,TypedDict,Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import  HumanMessage, SystemMessage
from langchain_community.tools.tavily_search import TavilySearchResults



## Defining Schemas

#### Task Model

In [2]:
class Task(BaseModel):
    id: int
    title: str
    goal: str = Field(..., description="One sentence describing what the user is able to do after this section")
    bullets: List[str] = Field(...,
                               min_length=3,
                                 max_length=6,
         description="3-6 concrete, non-overlapping subpoints to cover this section")
    target_words: int = Field(..., description="The target number of words for this section is (120-550)")
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False


#### Evidence Item Schema

In [3]:
class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at : Optional[str] = None # keep tavilyt provides, don't rely on it 
    source: Optional[str] = None # keep tavilyt provides, don't rely on it
    snippet: Optional[str] = None # keep tavilyt provides, don't rely on it

#### Evidence Pack Schema

In [4]:
class EvidencePack(BaseModel):
   evidence: List[EvidenceItem] = Field(default_factory=list)

#### Router Decision Schema

In [8]:
class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal['closed_book', 'hybrid', 'open_book']
    queries : List[str] = Field(default_factory=list)